In [ ]:
import numpy as np
import xarray as xr
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy
from glob import glob
from matplotlib import pyplot as plt
import matplotlib.colors as colors
import matplotlib.path as mpath
import matplotlib.ticker as mticker
import pandas as pd
import csv
from datetime import datetime
plt.rcParams['figure.facecolor'] = 'white'
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None}

In [ ]:
def prepro(ds):
    return ds.isel(y=slice(400, None))

Load grid and data files from CREG12

In [ ]:
grid_files = ["/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mask.nc", 
              "/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mesh_hgr.nc",
              "/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mesh_zgr.nc"]

In [ ]:
grid = xr.open_mfdataset(grid_files, parallel=True, preprocess=prepro)

In [ ]:
MKE_data_filesREF = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-REF08-MKE" 
                                + "/clim/CREG12.L75-REF08_*.5d_MKEclim.nc"))
EKE_data_filesREF = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-REF08-EKE/" 
                                + "/clim/CREG12.L75-REF08_*.5d_EKEclim.nc"))

In [ ]:
MKEREF = xr.open_mfdataset(MKE_data_filesREF, parallel=True, preprocess=prepro)
EKEREF = xr.open_mfdataset(EKE_data_filesREF, parallel=True, preprocess=prepro)

In [ ]:
MKEREF = MKEREF.assign_coords({"nav_lon": grid.nav_lon, "nav_lat": grid.nav_lat})
EKEREF = EKEREF.assign_coords({"nav_lon": grid.nav_lon, "nav_lat": grid.nav_lat})
MKEREFplot = MKEREF.vomke.sel(deptht=slice(0, 10)).mean("deptht").mean("time_counter").compute()
EKEREFplot = EKEREF.voeke.sel(deptht=slice(50, 100)).mean("deptht").mean("time_counter").compute()

Load Armitage (2017) dataset

In [ ]:
armitage = xr.open_dataset("CPOM_DOT.nc")

Convert dates to `datetime64` for easier handling

In [ ]:
armitage["date"] = np.array([np.datetime64(datetime(int(str(armitage.date[s].values)[0:4]), 
                                                    int(str(armitage.date[s].values)[4:6].lstrip("0")), 
                                                    15)) for s in range(0, len(armitage.date))])

In [ ]:
armitage = armitage.rename({"date": "time"})

Compute mean $u$ and $v$ from Armitage dataset, the same way the mean is computed form CREG12 (including all frequencies from seasonal cycle to longer)

In [ ]:
aUbar = ((armitage.U - armitage.U).groupby("time.year") 
         + armitage.U.groupby("time.year").mean()).groupby("time.month") + armitage.U.groupby("time.month").mean()
aVbar = ((armitage.V - armitage.V).groupby("time.year") 
         + armitage.V.groupby("time.year").mean()).groupby("time.month") + armitage.V.groupby("time.month").mean()

Compute MKE for Armitage dataset and interpolate it to the same grid as CREG12

In [ ]:
armitage["MKE"] = 0.5 * (aUbar**2 + aVbar**2)
aMKE = armitage.MKE.interp(lon=a2.lons, lat=a2.lats)

In [ ]:
aMKE = aMKE.assign_coords({"lon": MKREF.nav_lon, "lat": MKREF.nav_lat})

Load von Appen (2022) EKE dataset

In [ ]:
vonAppen = pd.read_csv("vonAppen-etal_2022.tab", sep="\t", header=300, quoting=csv.QUOTE_ALL).to_xarray()

Define some modifications to be made to all the maps plotted

In [ ]:
def map_config(ax):
    ax.set_extent([-180, 180, 67, 90], ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="silver")
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False, y_inline=True,
                     linewidth=1, color='gray', alpha=0.7, linestyle='--')
    b11 = bathyplot.plot.contour(x='nav_lon', y='nav_lat', levels=[1500], 
                             colors=["snow"], linewidths=1, alpha=0.8, linestyles="-", 
                             ax=ax, transform=ccrs.PlateCarree(), zorder=2)
    b12 = bathyplot.plot.contour(x='nav_lon', y='nav_lat', levels=[1500], 
                             colors=["dimgray"], linewidths=0.5, linestyles="-", 
                             ax=ax, transform=ccrs.PlateCarree(), alpha=0.8, zorder=3)
    plt.draw()
    return

Define `cartopy` projection to be used on maps

In [ ]:
proj = ccrs.RotatedPole(pole_longitude=180, pole_latitude=40, central_rotated_longitude=0)

Define the bathymetry

In [ ]:
bathy = grid.gdept_1d.squeeze().isel(z=grid.mbathy.squeeze().compute())
bathy = bathy.assign_coords({"nav_lon": grid.nav_lon, "nav_lat": grid.nav_lat}).compute()

In [ ]:
bathyplot = xr.where(bathy > 500, bathy, np.nan)
bathyplot[1118:1165, 775:787] = np.nan # 1500
bathyplot[878:881, 849:851] = np.nan # 1500

Plot Fig. S2

In [ ]:
fig = plt.figure(figsize=(10, 4))
gs = fig.add_gridspec(2, 24, height_ratios=[1, 0.06])

# define axes
ax1 = fig.add_subplot(gs[0, 0:8], projection=proj)
ax2 = fig.add_subplot(gs[0, 8:16], projection=proj)
ax3 = fig.add_subplot(gs[0, 16:24], projection=proj)
axcb1 = fig.add_subplot(gs[1, 2:14])
axcb2 = fig.add_subplot(gs[1, 17:23])

# plot MKE from Armitage (2017)
p1 = aMKE.mean("time").plot(x='lon', y='lat', cmap=cmo.matter_r, 
                             norm=colors.LogNorm(vmin=4e-5, vmax=4e-2),
                             ax=ax1, transform=ccrs.PlateCarree(), rasterized=True, 
                             zorder=1, add_colorbar=False)

# plot MKE from REF
p2 = MKEREFplot.plot(x='nav_lon', y='nav_lat', cmap=cmo.matter_r, 
                    norm=colors.LogNorm(vmin=4e-5, vmax=4e-2),
                    ax=ax2, transform=ccrs.PlateCarree(), rasterized=True, 
                    zorder=1, add_colorbar=False)

# plot EKE from REF
p3 = EKEREFplot.plot(x='nav_lon', y='nav_lat', cmap=cmo.matter_r, 
                    norm=colors.LogNorm(vmin=1e-5, vmax=2e-2), 
                    ax=ax3, transform=ccrs.PlateCarree(), rasterized=True, 
                    zorder=1, add_colorbar=False)

# plot dots with von Appen (2022) EKE
p3s = ax3.scatter(vonAppen.Longitude, vonAppen.Latitude, c=vonAppen["EKE mean [m**2/s**2]"], cmap=cmo.matter_r, 
                  norm=colors.LogNorm(vmin=1e-5, vmax=2e-2), transform=ccrs.PlateCarree(), s=20,
                  edgecolor="#1c1c1c", linewidth=0.5, zorder=4)


# add labels, text, titles
[ax.text(-143, 65, t, fontsize=12, transform=ccrs.PlateCarree(), backgroundcolor="whitesmoke") 
 for ax, t in zip([ax1, ax2, ax3], 
                  ["(a)", "(b)", "(c)"])]

gl = [map_config(ax) for ax in [ax1, ax2, ax3]];


for ax in [ax1, ax2, ax3]:
    ax.text(-30, 85, r"85$^{\circ}$N", transform=ccrs.PlateCarree(), ha="center", fontsize=7)
    ax.text(-31, 80, r"80$^{\circ}$N", transform=ccrs.PlateCarree(), ha="center", fontsize=7)
    ax.text(-34, 75, r"75$^{\circ}$N", transform=ccrs.PlateCarree(), ha="center", fontsize=7)
    ax.text(-37, 70, r"70$^{\circ}$N", transform=ccrs.PlateCarree(), ha="center", fontsize=7)
    ax.text(-41, 65, r"65$^{\circ}$N", transform=ccrs.PlateCarree(), ha="center", fontsize=7)
    
# add colorbars
cb1 = plt.colorbar(p1, cax=axcb1, orientation="horizontal", extend="both")
axcb1.set_xticks([1e-4, 1e-3, 1e-2])
axcb1.set_xticklabels([r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$"], rotation=45, ha='center')
axcb1.tick_params(axis='x', which='major', pad=5)
axcb1.set_xlabel(r"m$^{2}\,$s$^{-2}$")
cb2 = plt.colorbar(p3, cax=axcb2, orientation="horizontal", extend="both")
axcb2.set_xticks([1e-5, 1e-4, 1e-3, 1e-2])
axcb2.set_xticklabels([r"$10^{-5}$", r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$"], rotation=45, ha='center')
axcb2.tick_params(axis='x', which='major', pad=5)
axcb2.set_xlabel(r"m$^{2}\,$s$^{-2}$")

ax1.set_title("MKE (Armitage, 2017)", fontsize=16, fontweight="bold", pad=5)
ax2.set_title("MKE (REF)", fontsize=16, fontweight="bold", pad=5)
ax3.set_title("EKE", fontsize=16, fontweight="bold", pad=5)

plt.subplots_adjust(wspace=0.2, hspace=-0.1, left=0.07, top=0.9, bottom=0.2)

plt.savefig("figures/Figure_S2_MKE_EKE_obs.png", dpi=600)